# Template Run - BLIP-2 Fusion VQA

Train and evaluate one EXP on the shared full HDF5 cache with the subset sizes set in its EXP config.

## 1. Mount Drive and load repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = "/content/blip2-fusion-experiment-vqa"
GITHUB_USER = "<username>"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/{GITHUB_USER}/blip2-fusion-experiment-vqa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log --oneline -3

## 2. Install dependencies

In [ ]:
!pip install -r requirements.txt -q

import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

import wandb
wandb.login()

## 3. Choose run

In [ ]:
EXP_ID = "01"
RUN_NUMBER = "1"
YOUR_NAME = "ten"
DATA_ROOT = "/content/drive/MyDrive/blip2_project"

CONFIG_FILE = f"configs/exp{EXP_ID}.yaml"
RUN_NAME = f"exp{EXP_ID}_lan{RUN_NUMBER}_{YOUR_NAME}"
CACHE_DIR_NAME = "cache"
CACHE_DIR = f"{DATA_ROOT}/{CACHE_DIR_NAME}"
ANSWER_LIST = f"{DATA_ROOT}/data/ans2idx.json"
OUTPUT_DIR = f"{DATA_ROOT}/checkpoints/{RUN_NAME}"
EVAL_OUTPUT = f"{OUTPUT_DIR}/val_predictions.json"
CHECKPOINT = f"{OUTPUT_DIR}/best_model.pth"

print("Config:", CONFIG_FILE)
print("Run:", RUN_NAME)
print("Cache:", CACHE_DIR)
print("Output:", OUTPUT_DIR)

## 4. Check full cache

In [ ]:
import os
import h5py

train_h5 = f"{CACHE_DIR}/train_features.h5"
val_h5 = f"{CACHE_DIR}/val_features.h5"
for path in (train_h5, val_h5):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing full cache artifact: {path}")

with h5py.File(train_h5, "r") as f:
    train_key = next(iter(f.keys()))
    print("Train cache images:", len(f.keys()), "sample:", f[train_key].shape, f[train_key].dtype)
with h5py.File(val_h5, "r") as f:
    val_key = next(iter(f.keys()))
    print("Val cache images:", len(f.keys()), "sample:", f[val_key].shape, f[val_key].dtype)
print("Question subset sizes come from:", CONFIG_FILE)


## 5. Train

In [ ]:
!python scripts/train.py \
    --config "{CONFIG_FILE}" \
    --run_name "{RUN_NAME}" \
    --data_root "{DATA_ROOT}" \
    --cache_dir "{CACHE_DIR_NAME}" \
    --answer_list "{ANSWER_LIST}" \
    --output_dir "{OUTPUT_DIR}"

## 6. Evaluate

In [ ]:
!python scripts/evaluate.py \
    --config "{CONFIG_FILE}" \
    --checkpoint "{CHECKPOINT}" \
    --split val \
    --data_root "{DATA_ROOT}" \
    --cache_dir "{CACHE_DIR_NAME}" \
    --answer_list "{ANSWER_LIST}" \
    --output "{EVAL_OUTPUT}"

## 7. Resume

In [ ]:
!python scripts/train.py \
    --config "{CONFIG_FILE}" \
    --run_name "{RUN_NAME}" \
    --data_root "{DATA_ROOT}" \
    --cache_dir "{CACHE_DIR_NAME}" \
    --answer_list "{ANSWER_LIST}" \
    --output_dir "{OUTPUT_DIR}" \
    --resume auto